# Day 041 Project: Chat with your CSV

## What You're Building

A natural-language interface for any CSV: you ask a plain-English question, the LLM writes the pandas code, you execute it, and get the answer back.

**Deliverable:** Ask 4 different questions about SALES_DF. All `_run_project_checks()` pass.

## Project Requirements

1. Load `RETAIL_CSV` into `SALES_DF` with a `revenue` column
2. Ask: 'What is the total revenue?' → store result in `a1`
3. Ask: 'Which product has the highest revenue?' → store result in `a2`
4. Ask: 'How many orders are there in each region?' → store result in `a3`
5. Ask a question of your own choosing → store result in `a4`

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import re
import ollama
import pandas as pd
import io


def get_df_schema(df) -> str:
    lines = [f"Shape: {df.shape[0]} rows x {df.shape[1]} columns"]
    lines.append("\nColumns and dtypes:")
    for col, dtype in df.dtypes.items():
        lines.append(f"  {col}: {dtype}")
    lines.append(f"\nSample (first 3 rows):\n{df.head(3).to_string(index=False)}")
    return "\n".join(lines)


def build_query_prompt(question: str, schema_str: str) -> str:
    return (
        "You are a Python data analyst. Write pandas code to answer the question.\n\n"
        "Requirements:\n"
        "- The DataFrame is already loaded as `df`. `pd` is also in scope.\n"
        "- Store the final answer in a variable named `result`.\n"
        "- Respond with ONLY a fenced Python code block, no explanation.\n\n"
        f"DataFrame schema:\n{schema_str}\n\n"
        f"Question: {question}"
    )


import re

def extract_code(response: str) -> str:
    fence = '`' * 3
    match = re.search(fence + r'python\s*(.*?)' + fence, response, re.DOTALL)
    if match:
        return match.group(1).strip()
    match = re.search(fence + r'\s*(.*?)' + fence, response, re.DOTALL)
    if match:
        return match.group(1).strip()
    return response.strip()


import pandas as pd

def run_pandas_code(code: str, df) -> str:
    namespace = {'df': df, 'pd': pd}
    try:
        exec(code, namespace)
    except Exception as e:
        return f"Code execution error: {e}"
    result = namespace.get('result', 'No result variable found')
    return str(result)


import re
import ollama
import pandas as pd

def ask_df(df, question: str, model: str = 'llama3.2') -> str:
    schema  = get_df_schema(df)
    prompt  = build_query_prompt(question, schema)
    resp    = ollama.chat(model=model,
                          messages=[{"role": "user", "content": prompt}])
    code    = extract_code(resp["message"]["content"])
    return run_pandas_code(code, df)


RETAIL_CSV = (
    'order_id,product,category,region,price,quantity\n'
    '1,Widget,Electronics,North,25.0,10\n'
    '2,Gadget,Electronics,South,150.0,3\n'
    '3,Widget,Electronics,South,25.0,5\n'
    '4,Doohickey,Accessories,East,8.0,50\n'
    '5,Gadget,Electronics,East,150.0,7\n'
    '6,Widget,Electronics,East,25.0,4\n'
    '7,Doohickey,Accessories,North,8.0,20\n'
    '8,Gadget,Electronics,North,150.0,2\n'
    '9,Widget,Electronics,West,25.0,6\n'
    '10,Doohickey,Accessories,South,8.0,15\n'
    '11,Thingamajig,Accessories,North,200.0,1\n'
    '12,Thingamajig,Accessories,East,200.0,4'
)
SALES_DF = pd.read_csv(io.StringIO(RETAIL_CSV))
SALES_DF['revenue'] = SALES_DF['price'] * SALES_DF['quantity']

print(f'Loaded {SALES_DF.shape[0]} rows')

## Your Questions

In [ ]:
# Question 1: total revenue
# TODO: a1 = ask_df(SALES_DF, 'What is the total revenue?')
# TODO: print('Q1:', a1)

# Question 2: top product by revenue
# TODO: a2 = ask_df(SALES_DF, 'Which product has the highest revenue?')
# TODO: print('Q2:', a2)

# Question 3: orders per region
# TODO: a3 = ask_df(SALES_DF, 'How many orders are there in each region?')
# TODO: print('Q3:', a3)

# Question 4: your own question
# TODO: a4 = ask_df(SALES_DF, 'YOUR QUESTION HERE')
# TODO: print('Q4:', a4)

## Project Checks

In [ ]:
def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: SALES_DF loaded with revenue
    try:
        assert 'SALES_DF' in globals() and 'revenue' in SALES_DF.columns
        passed += 1; print(f'\u2705 Check 1: SALES_DF loaded ({len(SALES_DF)} rows)')
    except Exception as e:
        print(f'\u274c Check 1: {e}')

    # Check 2: a1 is a non-empty string containing total revenue
    try:
        assert 'a1' in globals(), 'a1 not defined'
        assert isinstance(a1, str) and len(a1.strip()) > 0
        assert '4105' in a1, f'a1 should contain 4105, got {repr(a1)}'
        passed += 1; print(f'\u2705 Check 2: a1 correct ({repr(a1[:40])})')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: a2 is a non-empty string mentioning Gadget
    try:
        assert 'a2' in globals(), 'a2 not defined'
        assert isinstance(a2, str) and len(a2.strip()) > 0
        assert 'Gadget' in a2 or 'gadget' in a2.lower(), \
            f'a2 should mention Gadget (highest revenue), got {repr(a2)}'
        passed += 1; print(f'\u2705 Check 3: a2 identifies Gadget as top product')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: a3 is a non-empty string
    try:
        assert 'a3' in globals(), 'a3 not defined'
        assert isinstance(a3, str) and len(a3.strip()) > 0
        passed += 1; print(f'\u2705 Check 4: a3 is non-empty ({len(a3)} chars)')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: a4 is a non-empty string (user's own question)
    try:
        assert 'a4' in globals(), 'a4 not defined'
        assert isinstance(a4, str) and len(a4.strip()) > 0
        passed += 1; print(f'\u2705 Check 5: a4 answered ({len(a4)} chars)')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Loop `ask_df` over a list of 10 questions and print all answers
- Add a retry: if the answer contains 'error', call `ask_df` again with 'make sure to assign the result to a variable called result' appended
- Display the generated pandas code alongside the answer (split `ask_df` into schema+prompt+LLM+extract steps and print the code before running it)
- On Day 43 you will build the SQL equivalent: ask questions and get SQL generated and executed against a real database